# **Part 5: Differential Expression Analysis**

---

## **Table of Contents**

---

## **Preliminary Setup**

In [ ]:
# automatically re-import custom modules
%reload_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import sys

# Resolve the absolute root directory of the project
project_root = Path.cwd().parent.resolve()

# Prepend project root to Python's search path to prioritize local module imports
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Define the directory path for intermediate outputs
intermediate_dir = project_root / "results" / "intermediates"

# Define absolute system paths for the CCLE data subsets
counts_filepath = intermediate_dir / "1_ccle_counts_subset.csv"
meta_filepath = intermediate_dir / "1_ccle_meta_subset.csv"

---

## **Import Packages**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage
from scipy.cluster.hierarchy import dendrogram, fcluster
from PyComplexHeatmap import ClusterMapPlotter, HeatmapAnnotation, anno_simple
from sklearn.decomposition import PCA

from src.utils.helpers import prepare_vst_data

In [ ]:
%matplotlib inline

---

## **Read and Prepare VST Data**

In the previous [notebook](./3_pca_analysis.ipynb), we saved the variance-stabilized transformation (VST) data as an `.h5ad` file (`1_ccle_vst_processed.h5ad`). For this session, we will simply read this pre-computed object back into memory. Alternatively, we could regenerate the `dds` object from scratch by rerunning the pipeline steps sourced by `prepare_vst_data` helper function and the original `ccle_counts_subset` and `ccle_meta_subset` dataframes as its inputs:

In [ ]:
# Define caching path and execution flag
vst_filepath = intermediate_dir / '1_ccle_vst.h5ad'
use_cached_data = True 

# Load cached VST data if available and requested; otherwise, compute from scratch
if vst_filepath.exists() and use_cached_data:
    print(f"Loading cached DeseqDataSet from: {vst_filepath}")
    dds = sc.read_h5ad(vst_filepath)
else:
    print("Cache missing or stale. Running PyDeseq2 VST pipeline...")
    dds = prepare_vst_data(counts_filepath, meta_filepath, vst_filepath=vst_filepath)

Let's verify the `dds` object and create `vst_df`:

In [ ]:
print(dds)
print(dds.shape)

In [ ]:
# Extract the transformed VST counts into a structured pandas DataFrame for analysis
vst_df = pd.DataFrame(
    dds.layers['vst_counts'],
    index=dds.obs_names,   # Sample IDs
    columns=dds.var_names  # Gene Names
)
vst_df.head()

---

## **Summary**